# Personalization-Placement Ablation — Analysis (post-split judges, 6-variant grid)

**The analysis notebook for this experiment — built and ready, waiting on data.** Every section is wired
up; it's empty right now only because the post-split re-run/re-score hasn't been done yet. Once it has,
**set `RESULTS_DIR` below and Run-All** and every section fills in automatically. It loads only post-split
results and never mixes in old / pre-split / relabeled runs.

### What "post-split" means here
The final-response judge is now **two** judges, merged into one `scores` dict:
- **Answer-Quality** (rubric + answer, *blind to evidence*): `intent_satisfaction`, `personalization_target_use`,
  `overpersonalization`, `specificity`, `safety`, `non_genericness`, `domain_safety`,
  `missing_constraint_awareness`, `actionability_without_overclaiming`, `uncertainty_calibration`.
- **Evidence-Faithfulness** (evidence + answer, *blind to persona/rubric*): `groundedness`,
  `unsupported_claim_risk`, `contradiction_with_evidence`, `citation_support`, `evidence_usage_quality`.
- Plus the **Retrieval** judge (7) and **Fan-out** judge (6) score files.

### The 6-variant grid
| | generic synthesis | personalized synthesis |
|---|---|---|
| **generic fan-out** | V1 | V2 |
| **personalized fan-out** | **V3** fanout_only | **V4** personalized_fanout |

(+ V0 single baseline, V5 mixed.) After the re-run all six exist, so the cleanest tests — `V3−V1`
(fan-out-only) and the fan-out×synthesis **interaction** — become computable.

### How to populate
1. Re-run the agent for all 6 variants and re-score with the current split judges (writing to `RESULTS_DIR`).
2. Set `RESULTS_DIR` / `RUNS_PATH` in the next cell, **Run All**.

In [ ]:
import os, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"]=(10,6); plt.rcParams["axes.edgecolor"]="#cccccc"; plt.rcParams["axes.linewidth"]=0.8
np.random.seed(0)

PROJECT_ROOT=os.path.dirname(os.getcwd()); OUT=os.path.join(PROJECT_ROOT,"outputs")
# ----------------------------------------------------------------------------------------------------
# SET THIS to the directory your post-split re-run/re-score wrote to. It must contain:
#   final_response_scores.jsonl, retrieval_scores.jsonl, fanout_scores.jsonl, and (for macro_domain) runs.jsonl
RESULTS_DIR = os.path.join(OUT, "placement_ablation_v1_postsplit")   # <-- does not exist yet; create via re-run
RUNS_PATH   = os.path.join(RESULTS_DIR, "runs.jsonl")
# ----------------------------------------------------------------------------------------------------

VARIANTS=["V0_generic_single","V1_generic_fanout","V2_synthesis_only_personalization",
          "V3_fanout_only_personalization","V4_personalized_fanout","V5_mixed_fanout"]
VLABEL={v:v.split("_")[0] for v in VARIANTS}; SHORT=[VLABEL[v] for v in VARIANTS]; VKEY={k.split("_")[0]:k for k in VARIANTS}
VCOLORS={"V0_generic_single":"#cccccc","V1_generic_fanout":"#999999",
         "V2_synthesis_only_personalization":"#4477aa","V3_fanout_only_personalization":"#ee8866",
         "V4_personalized_fanout":"#44bb99","V5_mixed_fanout":"#aa4499"}

QUALITY_METRICS=["intent_satisfaction","personalization_target_use","overpersonalization","specificity","safety",
                 "non_genericness","domain_safety","missing_constraint_awareness","actionability_without_overclaiming","uncertainty_calibration"]
FAITH_METRICS=["groundedness","unsupported_claim_risk","contradiction_with_evidence","citation_support","evidence_usage_quality"]
RETR_METRICS=["evidence_relevance","result_persona_fit","constraint_coverage","distractor_robustness","source_quality","disconfirming_coverage","unsafe_or_overpersonalized_retrieval_risk"]
FANOUT_METRICS=["persona_field_use","query_specificity","query_diversity","search_realism","faithfulness_to_user_query","overpersonalization_risk"]
LOWER_IS_BETTER={"overpersonalization","unsupported_claim_risk","contradiction_with_evidence",
                 "unsafe_or_overpersonalized_retrieval_risk","overpersonalization_risk"}

def _read(name, keys):
    p=os.path.join(RESULTS_DIR,name); out={}
    if not os.path.exists(p): return out
    for l in open(p):
        if not l.strip(): continue
        d=json.loads(l); s=d.get("scores") or {}
        if not s: continue
        out[d["run_id"]]=dict(variant=d.get("variant"),task_type=d.get("task_type"),query_id=d.get("query_id"),
                              **{k:s.get(k) for k in keys})
    return out

def load():
    fin=_read("final_response_scores.jsonl", QUALITY_METRICS+FAITH_METRICS)
    if not fin: return pd.DataFrame()
    retr=_read("retrieval_scores.jsonl", RETR_METRICS); fan=_read("fanout_scores.jsonl", FANOUT_METRICS)
    meta={}
    if os.path.exists(RUNS_PATH):
        for l in open(RUNS_PATH):
            if l.strip(): r=json.loads(l); meta[r["run_id"]]=r.get("macro_domain")
    rows=[]
    for rid,f in fin.items():
        rows.append(dict(run_id=rid, macro_domain=meta.get(rid),
                         **f,
                         **{k:retr.get(rid,{}).get(k) for k in RETR_METRICS},
                         **{k:fan.get(rid,{}).get(k) for k in FANOUT_METRICS}))
    df=pd.DataFrame(rows); df["v"]=df["variant"].map(VLABEL); return df

DF=load(); HAS_DATA=not DF.empty
if HAS_DATA:
    print("Loaded post-split scores:", len(DF), "rows")
    print("variants present:", [VLABEL[v] for v in VARIANTS if v in set(DF.variant)],
          "| missing:", [VLABEL[v] for v in VARIANTS if v not in set(DF.variant)])
else:
    print(f"⏳ No post-split scores found in {RESULTS_DIR}.")
    print("   Run the 6-variant re-run + re-score (current split judges) into that dir, then Run-All.")

WAIT="⏳ No post-split data yet — populate RESULTS_DIR (re-run + re-score), then re-run this cell."

def paired(a, b, metric, tt=None, B=5000, seed=0):
    """Paired per-query Δ(a-b) ± bootstrap 95% CI. Per-call seed => order-independent.
    status: ok | pending (variant absent) | empty (no overlapping non-NaN queries)."""
    if DF.empty: return dict(mean=np.nan,lo=np.nan,hi=np.nan,n=0,status="nodata")
    sub=DF if tt is None else DF[DF.task_type==tt]; va,vb=VKEY[a],VKEY[b]
    if va not in set(sub.variant) or vb not in set(sub.variant): return dict(mean=np.nan,lo=np.nan,hi=np.nan,n=0,status="pending")
    pa=sub[sub.variant==va].groupby("query_id")[metric].mean(); pb=sub[sub.variant==vb].groupby("query_id")[metric].mean()
    common=pa.index.intersection(pb.index); d=(pa.loc[common]-pb.loc[common]).values; d=d[~np.isnan(d)]; n=len(d)
    if n==0: return dict(mean=np.nan,lo=np.nan,hi=np.nan,n=0,status="empty")
    rng=np.random.default_rng(seed); boot=np.array([rng.choice(d,n,replace=True).mean() for _ in range(B)])
    return dict(mean=float(d.mean()),lo=float(np.percentile(boot,2.5)),hi=float(np.percentile(boot,97.5)),n=n,status="ok")

## Section 0 — Data coverage / sanity

Confirms what loaded, n per (variant × task-type), and any errored rows. (Populates after the re-run.)

In [ ]:
if not HAS_DATA: print(WAIT)
else:
    print("n per (variant, task_type):")
    print(DF.groupby(["v","task_type"]).size().unstack(fill_value=0).reindex(SHORT))
    for fam,ms in [("Quality",QUALITY_METRICS),("Faithfulness",FAITH_METRICS),("Retrieval",RETR_METRICS),("Fan-out",FANOUT_METRICS)]:
        miss=[m for m in ms if m in DF and DF[m].isna().all()]
        print(f"  {fam}: {len(ms)} metrics" + (f"  [MISSING: {miss}]" if miss else ""))

## Section 1 — The 2×2 placement grid

Down a column = fan-out personalization; right along a row = synthesis personalization. With V3 present
post-run, all four cells fill.

In [ ]:
if not HAS_DATA: print(WAIT)
else:
    def grid(metric):
        m=DF.groupby("variant")[metric].mean()
        M=pd.DataFrame(index=["generic fan-out","personalized fan-out"],columns=["generic synthesis","personalized synthesis"],dtype=float)
        M.iloc[0,0]=m.get("V1_generic_fanout",np.nan); M.iloc[0,1]=m.get("V2_synthesis_only_personalization",np.nan)
        M.iloc[1,0]=m.get("V3_fanout_only_personalization",np.nan); M.iloc[1,1]=m.get("V4_personalized_fanout",np.nan)
        return M
    hm=["intent_satisfaction","result_persona_fit","groundedness"]
    fig,axes=plt.subplots(1,3,figsize=(19,5.6))
    for ax,metric in zip(axes,hm):
        sns.heatmap(grid(metric), annot=True, fmt=".2f", cmap="viridis", vmin=1, vmax=5, linewidths=2, linecolor="white",
                    cbar_kws={"label":"mean (1-5)"}, annot_kws={"fontsize":13,"fontweight":"bold"}, ax=ax)
        ax.set_title(metric.replace("_"," ").title(), fontsize=12, fontweight="bold")
        ax.set_xlabel("→ synthesis personalization"); ax.set_ylabel("↓ fan-out personalization")
    plt.suptitle("2×2 placement grid (post-split): gain from fan-out (rows) vs synthesis (cols)?", y=1.05, fontsize=12, fontweight="bold")
    plt.tight_layout(); plt.show()

## Section 2 — Marginal effects + the interaction (full decomposition)

Paired Δ ± bootstrap 95% CI. With V3 present, the cleanest fan-out test (`V3−V1`) and the
**interaction** `(V4−V2) − (V3−V1)` (does fan-out help *more* when synthesis is also personalized?) are now
computable. CIs are uncorrected for multiple comparisons → directional.

In [ ]:
if not HAS_DATA: print(WAIT)
else:
    HEAD="intent_satisfaction"
    CONTR=[("V2","V1","synthesis pers. | generic fan-out"),
           ("V4","V3","synthesis pers. | pers. fan-out"),
           ("V3","V1","fan-out pers. | generic synth"),
           ("V4","V2","fan-out pers. | pers. synth"),
           ("V4","V1","full personalization"),
           ("V5","V4","mixed vs personalized fan-out")]
    fig,ax=plt.subplots(figsize=(11,6))
    for yi,(a,b,nm) in enumerate(CONTR):
        r=paired(a,b,HEAD)
        if r["status"]!="ok": ax.text(0.02,yi,r["status"],va="center",color="#cc3333",fontsize=9,fontweight="bold")
        else:
            ax.errorbar(r["mean"],yi,xerr=[[r["mean"]-r["lo"]],[r["hi"]-r["mean"]]],fmt="o",color="#4477aa",capsize=4,ms=8)
            ax.text(r["hi"]+0.04,yi,f"{r['mean']:+.2f}",va="center",fontsize=9)
    ax.axvline(0,color="#888",ls="--"); ax.set_yticks(range(len(CONTR))); ax.set_yticklabels([f"{a}-{b}  {nm}" for a,b,nm in CONTR]); ax.invert_yaxis()
    ax.set_title(f"Marginal effects on {HEAD} (paired Δ ± 95% CI; uncorrected → directional)", fontsize=11, fontweight="bold")
    ax.set_xlabel("mean_a - mean_b"); plt.tight_layout(); plt.show()
    fo_ps=paired("V4","V2",HEAD); fo_gs=paired("V3","V1",HEAD)
    if fo_ps["status"]=="ok" and fo_gs["status"]=="ok":
        print(f"Fan-out × synthesis INTERACTION on {HEAD}: (V4-V2) - (V3-V1) = {fo_ps['mean']-fo_gs['mean']:+.2f}")
        print("  >0 => fan-out personalization helps MORE when synthesis is also personalized (super-additive).")

## Section 3 — Task-type sensitivity

Each lever's effect on retrieval-sensitive vs synthesis-sensitive tasks (95% CIs).

In [ ]:
if not HAS_DATA: print(WAIT)
else:
    effects=[("V3","V1","fan-out\n(V3-V1)"),("V2","V1","synthesis\n(V2-V1)"),("V4","V1","both\n(V4-V1)")]
    metrics_tt=["intent_satisfaction","result_persona_fit"]; x=np.arange(len(effects)); w=0.35
    fig,axes=plt.subplots(1,2,figsize=(16,6))
    for ax,metric in zip(axes,metrics_tt):
        for k,tt in enumerate(["retrieval_sensitive","synthesis_sensitive"]):
            rs=[paired(a,b,metric,tt=tt) for a,b,_ in effects]
            means=[r["mean"] for r in rs]
            lo=[(r["mean"]-r["lo"]) if r["status"]=="ok" else 0 for r in rs]; hi=[(r["hi"]-r["mean"]) if r["status"]=="ok" else 0 for r in rs]
            ax.bar(x+(k-0.5)*w,[0 if pd.isna(m) else m for m in means],w,yerr=[lo,hi],capsize=3,label=tt,color=["#4477aa","#ee8866"][k],edgecolor="#555",error_kw=dict(elinewidth=1))
        ax.axhline(0,color="#888",ls="--"); ax.set_xticks(x); ax.set_xticklabels([e[2] for e in effects])
        ax.set_title(f"{metric.replace('_',' ').title()} — marginal Δ by task type", fontsize=12, fontweight="bold"); ax.set_ylabel("paired Δ"); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()

## Section 4 — Per macro-domain (education / legal / finance)

Intent satisfaction by variant within each domain (95% CI; needs `runs.jsonl` in `RESULTS_DIR` for the domain join).

In [ ]:
if not HAS_DATA: print(WAIT)
elif DF["macro_domain"].isna().all(): print("macro_domain unavailable — put runs.jsonl in RESULTS_DIR.")
else:
    P=DF[DF.macro_domain.notna()]; metric="intent_satisfaction"
    doms=sorted(P.macro_domain.unique()); x=np.arange(len(doms)); w=0.14
    fig,ax=plt.subplots(figsize=(13,6))
    for i,v in enumerate(VARIANTS):
        sub=P[P.variant==v]
        means=[sub[sub.macro_domain==d][metric].mean() for d in doms]; errs=[1.96*sub[sub.macro_domain==d][metric].sem() for d in doms]
        ax.bar(x+(i-2.5)*w,[0 if pd.isna(m) else m for m in means],w,yerr=[0 if pd.isna(e) else e for e in errs],
               label=VLABEL[v],color=VCOLORS[v],edgecolor="#555",linewidth=0.5,capsize=2,error_kw=dict(elinewidth=0.8))
    ax.set_xticks(x); ax.set_xticklabels(doms); ax.set_ylim(0,5.5); ax.set_ylabel("intent_satisfaction (1-5)")
    ax.set_title("Intent satisfaction by macro-domain and variant (post-split, 95% CI)", fontsize=12, fontweight="bold")
    ax.legend(title="variant", ncol=6, fontsize=8, loc="upper center", bbox_to_anchor=(0.5,-0.08)); plt.tight_layout(); plt.show()

## Section 5 — Answer-Quality vs Evidence-Faithfulness (the split's payoff)

The judge split lets us see the two axes **independently**: does personalization raise *answer quality*
without costing *faithfulness to evidence*? Left: quality (`personalization_target_use`) vs faithfulness
(`groundedness`) per variant. Right: the new faithfulness sub-metrics by variant.

In [ ]:
if not HAS_DATA: print(WAIT)
else:
    fig,axes=plt.subplots(1,2,figsize=(18,6))
    ax=axes[0]
    qx=DF.groupby("variant")["personalization_target_use"].mean().reindex(VARIANTS)
    fy=DF.groupby("variant")["groundedness"].mean().reindex(VARIANTS)
    for v in VARIANTS:
        if pd.notna(qx[v]) and pd.notna(fy[v]):
            ax.scatter(qx[v],fy[v],s=140,color=VCOLORS[v],edgecolor="#333",zorder=3)
            ax.annotate(VLABEL[v],(qx[v],fy[v]),xytext=(5,5),textcoords="offset points",fontsize=10,fontweight="bold")
    ax.set_xlabel("Answer-Quality: personalization_target_use"); ax.set_ylabel("Faithfulness: groundedness")
    ax.set_title("Quality vs Faithfulness per variant\n(top-right = personalized AND grounded)", fontsize=11, fontweight="bold")
    ax.set_xlim(1,5.2); ax.set_ylim(1,5.2)
    ax=axes[1]
    fsub=["groundedness","citation_support","evidence_usage_quality","unsupported_claim_risk","contradiction_with_evidence"]
    fsub=[m for m in fsub if m in DF]; xf=np.arange(len(fsub)); w=0.13
    for i,v in enumerate(VARIANTS):
        vals=DF[DF.variant==v][fsub].mean().reindex(fsub)
        ax.bar(xf+(i-2.5)*w,[0 if pd.isna(x) else x for x in vals.values],w,label=VLABEL[v],color=VCOLORS[v],edgecolor="#555",linewidth=0.4)
    ax.set_xticks(xf); ax.set_xticklabels([m.replace("_","\n") for m in fsub],fontsize=8); ax.set_ylim(0,5.3)
    ax.set_title("Evidence-Faithfulness sub-metrics by variant\n(unsupported_claim / contradiction: lower=better)", fontsize=11, fontweight="bold")
    ax.legend(title="variant", ncol=6, fontsize=7, loc="upper center", bbox_to_anchor=(0.5,-0.12)); plt.tight_layout(); plt.show()

## Section 6 — How to read this & caveats

- **Trustworthy axes:** Answer-Quality (rubric, evidence-blind) and Evidence-Faithfulness (evidence,
  persona-blind) are now decoupled, so personalization gains (§1/§2) and any faithfulness cost (§5) are
  separable — that's the point of the split.
- **`overpersonalization` is still overloaded** — it fires on `should_not_use` violations, which generic
  variants commit by *under*-personalizing. Read it alongside the variant (a high value on V0/V1 means
  "ignored the user's constraints," not "over-did it"); consider splitting that metric in the judge.
- **Statistics:** CIs are bootstrap/normal-approx and **uncorrected for multiple comparisons**; mind the n
  printed in §0. The `paired()` noise check still applies — a contrast that *should* be ≈0 but isn't is a
  small-n warning, not signal.
- **This notebook loads ONLY post-split data** from `RESULTS_DIR`; it never mixes in old/pre-split/relabeled
  runs. Until that dir is populated, every section prints a wait notice.